# ĐỒ ÁN MÔN HỌC: XỬ LÝ ẢNH — NHẬN DIỆN CHẤT LƯỢNG GẠO
## Phân hệ 2: Phân đoạn và tách hạt gạo (Segmentation) - Huy Lê

Notebook này thực hiện tách hạt khỏi nền và tách các hạt tiếp xúc/dính nhau trên tập ảnh mẻ gạo đã qua tiền xử lý (`processed_dataset/2_batch_rice_images/gray_enhanced/`)[cite: 3, 4].

**Chuỗi xử lý thuần Xử lý ảnh:**
1. Phân ngưỡng tự động Otsu (`cv2.threshold`) + xử lý hình thái học (`cv2.morphologyEx`).
2. Tính biến đổi khoảng cách (`cv2.distanceTransform`) để xác định tâm/vùng lõi hạt[cite: 4].
3. Tách vùng hạt tiếp xúc nhau bằng Watershed (`cv2.watershed`)[cite: 4].
4. Đánh số và lọc nhiễu vùng hạt bằng `cv2.connectedComponentsWithStats`[cite: 4].
5. Tạo ROI, bounding box, gán `grain_id`, vẽ đường biên/nhãn và xuất bảng + dữ liệu đầu ra cho TV3[cite: 4].

In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN TƯƠNG ĐỐI (TỰ ĐỘNG NHẬN DIỆN GỐC DỰ ÁN)
# ==============================================================================
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()

# Dữ liệu đầu vào chính của TV2: nhánh ảnh xám đã tiền xử lý từ TV1
BATCH_GRAY_DIR = os.path.join(
    PROJECT_ROOT, "processed_dataset", "2_batch_rice_images", "gray_enhanced"
)
BATCH_CSV_PATH = os.path.join(
    PROJECT_ROOT, "processed_dataset", "2_batch_rice_images", "batch_images_list.csv"
)

# Thư mục đầu ra của TV2
SEG_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "segmented_dataset")
SEG_MASKS_DIR = os.path.join(SEG_OUTPUT_DIR, "masks")
SEG_ANNOTATED_DIR = os.path.join(SEG_OUTPUT_DIR, "annotated")
SEG_ROIS_DIR = os.path.join(SEG_OUTPUT_DIR, "rois")
SEG_INDEX_CSV = os.path.join(SEG_OUTPUT_DIR, "segmented_grains.csv")

SUPPORTED_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# ==============================================================================
# 2. THAM SỐ PHÂN ĐOẠN TRÊN ẢNH XÁM (TỐI ƯU CHO HẠT GẠO TRÊN NỀN TỐI/LOANG SÁNG)
# ==============================================================================
SEG_CONFIG = {
    "use_clahe": True,
    "clahe_clip_limit": 3.0,
    "clahe_grid_size": (8, 8),
    "use_tophat": True,
    "tophat_kernel_size": 31,  # Lớn hơn kích thước hạt để giữ trọn thân hạt
    # Kết hợp Otsu + Adaptive Threshold cho vùng sáng/lóa
    "use_adaptive_glare_assist": True,
    "adaptive_block_size": 31,
    "glare_high_threshold": 150,  # Khớp với vùng sáng loang/lóa
    "glare_assist_c": -2.0,  # Tùy theo nền sáng hơn hay tối hơn hạt sau chuẩn hóa
    # Hình thái học
    "morph_kernel_size": 3,
    "open_iter": 0,  # Hạn chế xóa hạt vỡ/nhỏ; dùng medianBlur/GaussianBlur khử nhiễu trước
    "close_iter": 1,  # Khép kín khe hở nhỏ trên thân hạt
    # Watershed
    "dist_smooth_ksize": 7,  # Làm mịn distance map để tránh xé lẻ 1 hạt thành nhiều mảnh
    "dist_foreground_ratio": 0.28,  # Giữ lõi hạt ổn định hơn, tránh làm mất hạt nhỏ
    "sure_bg_dilate_iter": 2,
    # Lọc hình dạng: loại đồng xu & nhiễu, giữ hạt nguyên & hạt vỡ
    "min_grain_area": 25,
    "max_grain_area": 900,  # Điều chỉnh theo kích thước hạt thực tế trên ảnh 800x800
    "min_aspect_ratio": 1.35,  # Loại vật thể tròn kiểu đồng xu (aspect_ratio ~ 1.0)
    "max_aspect_ratio": 7.5,
    "max_circularity": 0.78,  # Đồng xu có circularity ~ 0.85 - 1.0, hạt gạo thuôn dài sẽ thấp hơn
    "min_solidity": 0.80,
    "roi_padding": 5,
}

print(f"Thư mục gốc: {PROJECT_ROOT}")
print(f"Nguồn ảnh xám đầu vào: {BATCH_GRAY_DIR}")

In [10]:
def load_batch_gray_records(gray_dir: str, csv_path: str) -> list[dict]:
    """
    Nạp danh sách ảnh xám mẻ gạo từ batch_images_list.csv hoặc quét trực tiếp
    thư mục gray_enhanced.
    """
    records = []

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        for idx, row in df.iterrows():
            gray_path = str(row.get("gray_path", "")).strip()

            if gray_path and not os.path.isabs(gray_path):
                gray_path = os.path.join(PROJECT_ROOT, gray_path)

            if not gray_path or not os.path.exists(gray_path):
                fname = os.path.basename(gray_path) if gray_path else ""
                if not fname and "original_file" in row:
                    base_no_ext = os.path.splitext(str(row["original_file"]))[0]
                    for ext in SUPPORTED_EXTS:
                        candidate = os.path.join(gray_dir, base_no_ext + ext)
                        if os.path.exists(candidate):
                            gray_path = candidate
                            break
                else:
                    candidate = os.path.join(gray_dir, fname)
                    if os.path.exists(candidate):
                        gray_path = candidate

            if gray_path and os.path.exists(gray_path):
                batch_id = str(row.get("batch_id", f"batch_{idx+1:04d}"))
                records.append({
                    "batch_id": batch_id,
                    "gray_path": gray_path,
                })

    if not records and os.path.exists(gray_dir):
        files = sorted(
            f for f in os.listdir(gray_dir)
            if f.lower().endswith(SUPPORTED_EXTS)
        )
        for idx, fname in enumerate(files):
            records.append({
                "batch_id": os.path.splitext(fname)[0] or f"batch_{idx+1:04d}",
                "gray_path": os.path.join(gray_dir, fname),
            })

    return records



def preprocess_gray_for_threshold(gray_img: np.ndarray, cfg: dict) -> np.ndarray:
    """
    Tăng cường tương phản cục bộ (CLAHE) và chuẩn hóa nền bằng Top-Hat/Black-Hat
    để làm nổi hạt gạo ngay cả khi có vùng ánh sáng mạnh chiếu vào.
    """
    gray = gray_img if gray_img.ndim == 2 else cv2.cvtColor(gray_img, cv2.COLOR_BGR2GRAY)

    work = gray.copy()
    if cfg.get("use_clahe", True):
        clahe = cv2.createCLAHE(
            clipLimit=float(cfg.get("clahe_clip_limit", 3.5)),
            tileGridSize=tuple(cfg.get("clahe_grid_size", (8, 8))),
        )
        work = clahe.apply(work)

    if cfg.get("use_tophat", True):
        k_th = max(15, int(cfg.get("tophat_kernel_size", 23)))
        if k_th % 2 == 0:
            k_th += 1
        th_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_th, k_th))
        morph_op = cv2.MORPH_BLACKHAT if cfg.get("invert_if_dark_rice", False) else cv2.MORPH_TOPHAT
        work = cv2.morphologyEx(work, morph_op, th_kernel)

    return work


def threshold_and_morph(gray_img: np.ndarray, cfg: dict) -> tuple[np.ndarray, np.ndarray]:
    
    gray = gray_img if gray_img.ndim == 2 else cv2.cvtColor(gray_img, cv2.COLOR_BGR2GRAY)

    # Khử nhiễu nhẹ giữ biên trước khi phân ngưỡng
    denoised = cv2.medianBlur(gray, 3)

    work = denoised.copy()
    if cfg.get("use_clahe", True):
        clahe = cv2.createCLAHE(
            clipLimit=float(cfg.get("clahe_clip_limit", 3.0)),
            tileGridSize=tuple(cfg.get("clahe_grid_size", (8, 8))),
        )
        work = clahe.apply(work)

    if cfg.get("use_tophat", True):
        k_th = max(21, int(cfg.get("tophat_kernel_size", 31)))
        if k_th % 2 == 0:
            k_th += 1
        th_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_th, k_th))
        work = cv2.morphologyEx(work, cv2.MORPH_TOPHAT, th_kernel)

    # Phân ngưỡng Otsu
    _, binary_otsu = cv2.threshold(work, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    combined_mask = binary_otsu.copy()

    # Hỗ trợ vùng sáng mạnh / lóa
    if cfg.get("use_adaptive_glare_assist", True):
        block_size = max(21, int(cfg.get("adaptive_block_size", 31)))
        if block_size % 2 == 0:
            block_size += 1

        glare_thresh = int(cfg.get("glare_high_threshold", 150))
        glare_region = (gray >= glare_thresh).astype(np.uint8) * 255
        glare_region = cv2.dilate(
            glare_region,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)),
            iterations=1,
        )

        smooth_for_adapt = cv2.GaussianBlur(work, (5, 5), 0)
        adapt_mask = cv2.adaptiveThreshold(
            smooth_for_adapt,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            float(cfg.get("glare_assist_c", -2.0)),
        )

        glare_grains = cv2.bitwise_and(adapt_mask, glare_region)
        combined_mask = cv2.bitwise_or(combined_mask, glare_grains)

    k = max(3, int(cfg.get("morph_kernel_size", 3)))
    if k % 2 == 0:
        k += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

    open_iter = int(cfg.get("open_iter", 0))
    close_iter = int(cfg.get("close_iter", 1))

    clean_mask = combined_mask
    if open_iter > 0:
        clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_OPEN, kernel, iterations=open_iter)
    if close_iter > 0:
        clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)

    return clean_mask, work



def separate_touching_grains_watershed(binary_mask: np.ndarray, cfg: dict) -> np.ndarray:
    """
    Làm mịn distance transform để hạn chế tách nhầm hạt, sau đó tách hạt tiếp xúc bằng Watershed.
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    sure_bg = cv2.dilate(binary_mask, kernel, iterations=int(cfg.get("sure_bg_dilate_iter", 2)))

    dist_transform = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)
    smooth_k = int(cfg.get("dist_smooth_ksize", 5))
    if smooth_k >= 3:
        if smooth_k % 2 == 0:
            smooth_k += 1
        dist_transform = cv2.GaussianBlur(dist_transform, (smooth_k, smooth_k), 0)

    max_dist = float(dist_transform.max())
    if max_dist <= 0:
        return np.zeros_like(binary_mask, dtype=np.int32)

    fg_ratio = float(cfg.get("dist_foreground_ratio", 0.35))
    _, sure_fg = cv2.threshold(dist_transform, fg_ratio * max_dist, 255, cv2.THRESH_BINARY)
    sure_fg = sure_fg.astype(np.uint8)

    unknown = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers.astype(np.int32) + 1
    markers[unknown == 255] = 0

    ws_input = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(ws_input, markers)
    markers[binary_mask == 0] = 1

    return markers


def is_valid_grain_shape(single_mask: np.ndarray, area: int, w: int, h: int, cfg: dict) -> bool:
    min_area = int(cfg.get("min_grain_area", 25))
    max_area = int(cfg.get("max_grain_area", 900))
    if area < min_area or area > max_area:
        return False

    contours, _ = cv2.findContours(single_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False

    cnt = max(contours, key=cv2.contourArea)
    cnt_area = cv2.contourArea(cnt)
    if cnt_area <= 0:
        return False

    # 1. Tỷ lệ dài / rộng theo góc xoay thực tế (minAreaRect) -> loại vật thể tròn như đồng xu
    (_, _), (rect_w, rect_h), _ = cv2.minAreaRect(cnt)
    short_side = max(1e-5, min(rect_w, rect_h))
    long_side = max(rect_w, rect_h)
    aspect_ratio = long_side / short_side

    if aspect_ratio < float(cfg.get("min_aspect_ratio", 1.35)) or aspect_ratio > float(
        cfg.get("max_aspect_ratio", 7.5)
    ):
        return False

    # 2. Độ tròn (Circularity) -> đồng xu có giá trị gần 1.0, hạt gạo thuôn dài sẽ nhỏ hơn
    perimeter = cv2.arcLength(cnt, True)
    if perimeter > 0:
        circularity = 4.0 * np.pi * cnt_area / (perimeter * perimeter)
        if circularity > float(cfg.get("max_circularity", 0.78)):
            return False

    # 3. Độ đặc (Solidity)
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    if hull_area <= 0:
        return False

    solidity = float(cnt_area) / float(hull_area)
    if solidity < float(cfg.get("min_solidity", 0.80)):
        return False

    return True


def extract_grain_instances_gray(
    gray_img: np.ndarray,
    markers: np.ndarray,
    cfg: dict,
) -> tuple[np.ndarray, list[dict]]:
    """
    Trích xuất từng hạt gạo hợp lệ sau phân đoạn và lọc hình dạng.
    """
    h_img, w_img = gray_img.shape[:2]
    final_label_map = np.zeros((h_img, w_img), dtype=np.int32)
    grains = []

    unique_labels = [lbl for lbl in np.unique(markers) if lbl > 1]
    grain_counter = 0

    for lbl in unique_labels:
        region_mask = (markers == lbl).astype(np.uint8)
        num_cc, cc_labels, stats, centroids = cv2.connectedComponentsWithStats(region_mask, connectivity=8)

        for cc_id in range(1, num_cc):
            area = int(stats[cc_id, cv2.CC_STAT_AREA])
            x = int(stats[cc_id, cv2.CC_STAT_LEFT])
            y = int(stats[cc_id, cv2.CC_STAT_TOP])
            w = int(stats[cc_id, cv2.CC_STAT_WIDTH])
            h = int(stats[cc_id, cv2.CC_STAT_HEIGHT])

            single_mask = (cc_labels == cc_id).astype(np.uint8) * 255
            if not is_valid_grain_shape(single_mask, area, w, h, cfg):
                continue

            grain_counter += 1
            grain_id = grain_counter
            cx = float(centroids[cc_id][0])
            cy = float(centroids[cc_id][1])

            final_label_map[single_mask == 255] = grain_id

            contours, _ = cv2.findContours(single_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            main_contour = max(contours, key=cv2.contourArea) if contours else None

            pad = int(cfg.get("roi_padding", 5))
            x1 = max(0, x - pad)
            y1 = max(0, y - pad)
            x2 = min(w_img, x + w + pad)
            y2 = min(h_img, y + h + pad)

            roi_gray = gray_img[y1:y2, x1:x2].copy()
            roi_mask = single_mask[y1:y2, x1:x2].copy()

            grains.append({
                "grain_id": grain_id,
                "bbox_x": x,
                "bbox_y": y,
                "bbox_w": w,
                "bbox_h": h,
                "centroid_x": round(cx, 2),
                "centroid_y": round(cy, 2),
                "area_px": area,
                "contour": main_contour,
                "roi_gray": roi_gray,
                "roi_mask": roi_mask,
            })

    return final_label_map, grains


def draw_segmentation_on_gray(gray_img: np.ndarray, grains: list[dict]) -> np.ndarray:
    """
    Vẽ đường biên (contour), bounding box, tâm hạt và ID lên ảnh xám (hiển thị BGR).
    """
    vis = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2BGR)

    for g in grains:
        gid = g["grain_id"]
        x, y, w, h = g["bbox_x"], g["bbox_y"], g["bbox_w"], g["bbox_h"]
        cx, cy = int(round(g["centroid_x"])), int(round(g["centroid_y"]))

        if g["contour"] is not None:
            cv2.drawContours(vis, [g["contour"]], -1, (0, 255, 0), 1)

        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 220, 255), 1)
        cv2.circle(vis, (cx, cy), 2, (0, 0, 255), -1)

        label_y = y - 3 if y - 3 > 10 else y + h + 11
        cv2.putText(
            vis,
            f"{gid}",
            (x, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (255, 255, 0),
            1,
            cv2.LINE_AA,
        )

    return vis

In [ ]:
def prepare_clean_output_dirs(reset: bool = True) -> None:
    if reset and os.path.exists(SEG_OUTPUT_DIR):
        shutil.rmtree(SEG_OUTPUT_DIR)
    os.makedirs(SEG_MASKS_DIR, exist_ok=True)
    os.makedirs(SEG_ANNOTATED_DIR, exist_ok=True)
    os.makedirs(SEG_ROIS_DIR, exist_ok=True)


def run_segmentation_pipeline(cfg: dict = SEG_CONFIG, reset_output: bool = True) -> pd.DataFrame:
    prepare_clean_output_dirs(reset=reset_output)

    records = load_batch_gray_records(BATCH_GRAY_DIR, BATCH_CSV_PATH)
    if not records:
        raise FileNotFoundError(f"Không tìm thấy ảnh xám tại '{BATCH_GRAY_DIR}'.")

    all_rows = []
    print(f"Bắt đầu phân đoạn trên ảnh xám cho {len(records)} ảnh...")

    for rec in records:
        batch_id = rec["batch_id"]
        gray_path = rec["gray_path"]

        gray_img = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)
        if gray_img is None:
            print(f"[CẢNH BÁO] Không đọc được ảnh: {gray_path}")
            continue

        clean_mask, _ = threshold_and_morph(gray_img, cfg)
        markers = separate_touching_grains_watershed(clean_mask, cfg)
        label_map, grains = extract_grain_instances_gray(gray_img, markers, cfg)

        mask_out_path = os.path.join(SEG_MASKS_DIR, f"{batch_id}_mask.png")
        annotated_out_path = os.path.join(SEG_ANNOTATED_DIR, f"{batch_id}_segmented_gray.png")
        label_map_path = os.path.join(SEG_MASKS_DIR, f"{batch_id}_labels.npy")

        final_binary_mask = (label_map > 0).astype(np.uint8) * 255
        annotated_gray_vis = draw_segmentation_on_gray(gray_img, grains)

        cv2.imwrite(mask_out_path, final_binary_mask)
        cv2.imwrite(annotated_out_path, annotated_gray_vis)
        np.save(label_map_path, label_map)

        img_roi_dir = os.path.join(SEG_ROIS_DIR, batch_id)
        os.makedirs(img_roi_dir, exist_ok=True)

        for g in grains:
            gid = g["grain_id"]
            grain_code = f"{batch_id}_g{gid:04d}"

            roi_gray_path = os.path.join(img_roi_dir, f"{grain_code}_gray.png")
            roi_mask_path = os.path.join(img_roi_dir, f"{grain_code}_mask.png")
            cv2.imwrite(roi_gray_path, g["roi_gray"])
            cv2.imwrite(roi_mask_path, g["roi_mask"])

            all_rows.append({
                "grain_uid": grain_code,
                "batch_id": batch_id,
                "grain_id": gid,
                "bbox_x": g["bbox_x"],
                "bbox_y": g["bbox_y"],
                "bbox_w": g["bbox_w"],
                "bbox_h": g["bbox_h"],
                "centroid_x": g["centroid_x"],
                "centroid_y": g["centroid_y"],
                "area_px": g["area_px"],
                "roi_gray_path": os.path.relpath(roi_gray_path, PROJECT_ROOT),
                "roi_mask_path": os.path.relpath(roi_mask_path, PROJECT_ROOT),
                "label_map_path": os.path.relpath(label_map_path, PROJECT_ROOT),
                "annotated_gray_path": os.path.relpath(annotated_out_path, PROJECT_ROOT),
            })

        print(f"  - {batch_id}: tách được {len(grains)} hạt")

    df_result = pd.DataFrame(all_rows)
    df_result.to_csv(SEG_INDEX_CSV, index=False, encoding="utf-8-sig")
    print(f"\nHoàn tất! Tổng số hạt tách được: {len(df_result)}")
    return df_result


df_grains = run_segmentation_pipeline(SEG_CONFIG, reset_output=True)
df_grains.head(10)

In [ ]:
import matplotlib.pyplot as plt

target_batch = "batch_rice_0033"
if target_batch not in df_grains["batch_id"].values and len(df_grains) > 0:
    target_batch = df_grains["batch_id"].iloc[0]

mask_path = os.path.join(SEG_MASKS_DIR, f"{target_batch}_mask.png")
vis_path = os.path.join(SEG_ANNOTATED_DIR, f"{target_batch}_segmented_gray.png")

if os.path.exists(mask_path) and os.path.exists(vis_path):
    mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    vis_rgb = cv2.cvtColor(cv2.imread(vis_path), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(mask_img, cmap="gray")
    axes[0].set_title(f"Mask phân đoạn trên ảnh xám - {target_batch}")
    axes[0].axis("off")

    axes[1].imshow(vis_rgb)
    axes[1].set_title(f"Biên, tâm & ID trên ảnh xám - {target_batch}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()